In [ ]:
@tool
def calculate_tip(total_bill: float, tip_percent: float) -> float:
    """
     a new tool named calculate_tip that takes a total_bill as float and tip_percent as float, and returns the tip amount as float
     Arguments:
         total_bill: float type
         tip_percent: float type
     Return:
         tip_amount: float type
    """
    tip_amount = total_bill + (total_bill*tip_percent)
    return tip_amount


@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b


tools_map_tip = {
    "calculate_tip" : calculate_tip,
    "add" : add,
    "subtract" : subtract,
    "multiply" : multiply
}


tools_tip = [calculate_tip, add, subtract, multiply]

In [ ]:
# TODO: Exercise 2
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")


class TipAgent:
    def __init__(self, llm):
        self.llm_with_tool = llm.bind_tools(tools_tip)
        self.tool_map = tools_map_tip

    def run(self, query: str) -> str:
        chat_history = [HumanMessage(content=query)]
        response = self.llm_with_tool.invoke(chat_history)

        tool_calls = response.tool_calls
        tool_name = tool_calls[0]["name"]
        tool_args = tool_calls[0]["args"]
        tool_call_id = tool_calls[0]["id"]
        
        tool_response = self.tool_map[tool_name].invoke(tool_args)
        tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_id)
        
        chat_history.extend([response, tool_message])
        
        return self.llm_with_tool.invoke(chat_history).content

agent = TipAgent(llm)
agent.run("How much should I tip on $60 at 20%?")
            